<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/20_llm_eval_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Custom LLM Evaluation Suite

> **Track:** AI Engineer | **Level:** Advanced | **Time:** 3-4 hours

## Overview
You cannot improve what you cannot measure. This notebook builds a **complete LLM evaluation framework** from scratch — covering automated metrics, LLM-as-judge, regression testing, and evaluation reports.

### What You'll Learn
- BLEU and ROUGE for text generation
- BERTScore for semantic similarity
- LLM-as-judge for subjective quality
- Building a golden evaluation dataset
- Regression testing across prompt versions
- Generating shareable evaluation reports

```bash
pip install openai pandas numpy evaluate bert-score
```

In [ ]:
# Setup: evaluation dataset and helpers
import os
import json
import time
from dataclasses import dataclass, field
from typing import Any
import numpy as np
import pandas as pd
import openai

client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY', 'YOUR_KEY'))

@dataclass
class EvalCase:
    id: str
    prompt: str
    reference: str  # ideal answer
    generated: str = ''
    metadata: dict = field(default_factory=dict)

# Golden evaluation dataset (expand in practice to 50-100 cases)
GOLDEN_DATASET: list[EvalCase] = [
    EvalCase('q1', 'What is gradient descent?', 'Gradient descent is an optimization algorithm that iteratively moves parameters in the direction that reduces a loss function, using the negative gradient.'),
    EvalCase('q2', 'Explain the difference between precision and recall.', 'Precision is TP/(TP+FP): of predicted positives, how many are correct. Recall is TP/(TP+FN): of actual positives, how many were found.'),
    EvalCase('q3', 'What is a transformer attention mechanism?', 'Attention computes a weighted sum of values, where weights are based on compatibility between queries and keys, using scaled dot-product: softmax(QK^T/sqrt(d_k))V.'),
    EvalCase('q4', 'What is the vanishing gradient problem?', 'In deep networks, gradients shrink exponentially during backpropagation through many layers, making it hard to train early layers. Solved by ReLU, BatchNorm, and residual connections.'),
]

# Simulated model responses
MODEL_ANSWERS = {
    'q1': 'Gradient descent minimizes a function by moving in the direction of steepest descent, computing updates as: theta = theta - lr * gradient.',
    'q2': 'Precision measures accuracy of positive predictions (TP/TP+FP). Recall measures coverage of actual positives (TP/TP+FN). F1 balances them.',
    'q3': 'The transformer uses self-attention to compute relationships between all tokens in parallel, replacing recurrence.',
    'q4': 'Vanishing gradients occur in deep networks when backpropagation shrinks gradients to near zero, preventing weight updates in early layers.',
}
for case in GOLDEN_DATASET:
    case.generated = MODEL_ANSWERS[case.id]

print(f'Evaluation dataset: {len(GOLDEN_DATASET)} golden test cases')

## 1. Automated Text Metrics (ROUGE)

In [ ]:
# ROUGE and BLEU metrics for text similarity
try:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    ROUGE_AVAILABLE = True
except ImportError:
    ROUGE_AVAILABLE = False
    print('pip install rouge-score for ROUGE metrics')

def compute_rouge(reference: str, generated: str) -> dict[str, float]:
    """Compute ROUGE-1, ROUGE-2, ROUGE-L F1 scores."""
    if not ROUGE_AVAILABLE:
        # Fallback: simple overlap ratio
        ref_words = set(reference.lower().split())
        gen_words = set(generated.lower().split())
        overlap = len(ref_words & gen_words) / max(len(ref_words), 1)
        return {'rouge1': overlap, 'rouge2': overlap * 0.5, 'rougeL': overlap * 0.8}
    scores = scorer.score(reference, generated)
    return {k: round(v.fmeasure, 4) for k, v in scores.items()}

# Compute metrics for all test cases
print('ROUGE Scores:')
rouge_results = []
for case in GOLDEN_DATASET:
    scores = compute_rouge(case.reference, case.generated)
    scores['id'] = case.id
    rouge_results.append(scores)
    print(f'  [{case.id}] R1={scores["rouge1"]:.3f} R2={scores["rouge2"]:.3f} RL={scores["rougeL"]:.3f}')

rouge_df = pd.DataFrame(rouge_results)
print(f'\nMean ROUGE-L: {rouge_df["rougeL"].mean():.3f}')

## 2. LLM-as-Judge (G-Eval)

In [ ]:
# G-Eval: use GPT-4 as judge to score on helpfulness, accuracy, clarity
JUDGE_PROMPT = """You are an expert evaluator. Score the answer on 3 dimensions (1-5 each).

Question: {question}
Reference answer: {reference}
Student answer: {generated}

Return JSON with keys: accuracy (1-5), clarity (1-5), completeness (1-5), reasoning (str)"""

@dataclass
class JudgeScore:
    case_id: str
    accuracy: float
    clarity: float
    completeness: float
    reasoning: str

    @property
    def overall(self) -> float:
        return (self.accuracy + self.clarity + self.completeness) / 3

def llm_judge(case: EvalCase) -> JudgeScore:
    """Score an eval case using GPT as judge."""
    prompt = JUDGE_PROMPT.format(question=case.prompt, reference=case.reference, generated=case.generated)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'}
    )
    result = json.loads(response.choices[0].message.content)
    return JudgeScore(case_id=case.id, **result)

# Mock results for demo
MOCK_JUDGE = [
    JudgeScore('q1', accuracy=4.5, clarity=5.0, completeness=4.0, reasoning='Clear and accurate, missing learning rate detail.'),
    JudgeScore('q2', accuracy=5.0, clarity=4.5, completeness=5.0, reasoning='Excellent, covers both metrics with formulas.'),
    JudgeScore('q3', accuracy=3.5, clarity=4.0, completeness=3.0, reasoning='Conceptually correct but misses the math.'),
    JudgeScore('q4', accuracy=5.0, clarity=5.0, completeness=4.5, reasoning='Very good, could mention specific solutions.'),
]

print('LLM Judge Results:')
for s in MOCK_JUDGE:
    print(f'  [{s.case_id}] Accuracy={s.accuracy} Clarity={s.clarity} Completeness={s.completeness} Overall={s.overall:.2f}')
print(f'\nMean overall score: {np.mean([s.overall for s in MOCK_JUDGE]):.2f}/5.0')

## 3. Regression Testing Across Prompt Versions

In [ ]:
# Regression testing: compare v1 vs v2 prompt performance
from scipy import stats

@dataclass
class EvalRun:
    version: str
    timestamp: str
    scores: list[float] = field(default_factory=list)

    @property
    def mean(self) -> float:
        return float(np.mean(self.scores))

    @property
    def std(self) -> float:
        return float(np.std(self.scores))

def regression_check(baseline: EvalRun, candidate: EvalRun, alpha: float = 0.05) -> dict[str, Any]:
    """Statistical test: did the new prompt version improve or regress?"""
    t_stat, p_value = stats.ttest_rel(candidate.scores, baseline.scores)
    delta = candidate.mean - baseline.mean
    significant = p_value < alpha
    return {
        'baseline': {'version': baseline.version, 'mean': round(baseline.mean, 3), 'std': round(baseline.std, 3)},
        'candidate': {'version': candidate.version, 'mean': round(candidate.mean, 3), 'std': round(candidate.std, 3)},
        'delta': round(delta, 3),
        'p_value': round(p_value, 4),
        'significant': significant,
        'verdict': ('IMPROVED' if delta > 0 else 'REGRESSED') + ('' if significant else ' (not significant)')
    }

# Simulate two prompt versions
v1 = EvalRun('v1.0', '2024-01-15', scores=[4.0, 4.5, 3.5, 4.5])
v2 = EvalRun('v2.0', '2024-02-01', scores=[4.5, 5.0, 4.0, 4.5])

result = regression_check(v1, v2)
print('=== Regression Test ===')
print(json.dumps(result, indent=2))

## 4. Evaluation Report Generation

In [ ]:
# Generate a complete evaluation report
def generate_report(
    run: EvalRun,
    rouge_df: pd.DataFrame,
    judge_scores: list[JudgeScore],
    threshold: float = 3.5
) -> str:
    """Generate a markdown evaluation report."""
    judge_df = pd.DataFrame([
        {'id': s.case_id, 'accuracy': s.accuracy, 'clarity': s.clarity,
         'completeness': s.completeness, 'overall': s.overall}
        for s in judge_scores
    ])
    failing = judge_df[judge_df['overall'] < threshold]
    pass_rate = (len(judge_df) - len(failing)) / len(judge_df)

    report = f"""# LLM Evaluation Report
**Version:** {run.version} | **Date:** {run.timestamp} | **Cases:** {len(judge_scores)}

## Summary
| Metric | Score |
|--------|-------|
| Pass rate (>{threshold}/5) | {pass_rate:.0%} |
| Mean overall (judge) | {judge_df['overall'].mean():.2f}/5.0 |
| Mean ROUGE-L | {rouge_df['rougeL'].mean():.3f} |
| Mean accuracy (judge) | {judge_df['accuracy'].mean():.2f}/5.0 |

## Failing Cases
{failing[['id','overall']].to_string(index=False) if len(failing) > 0 else 'None — all cases passed!'}

## Recommendation
{'✅ APPROVE FOR DEPLOYMENT' if pass_rate >= 0.8 else '❌ DO NOT DEPLOY — Fix failing cases first'}
"""
    return report

report = generate_report(v2, rouge_df, MOCK_JUDGE)
print(report)

## Exercises

1. **BERTScore integration**: Install `bert-score` and compute BERTScore F1 for each eval case. Compare how BERTScore rankings differ from ROUGE rankings — which cases are ranked differently and why?

2. **Adversarial test cases**: Add 5 test cases designed to **break** your LLM: ambiguous questions, trick questions, questions outside the model's knowledge. Measure how often the model correctly says 'I don't know' vs. hallucinating a confident wrong answer.

3. **Automated dataset expansion**: Write a function that uses an LLM to automatically generate 20 more evaluation cases for a given topic (e.g., 'machine learning'). For each generated question, also generate the reference answer and a distractor (plausible wrong answer). Build a mini benchmark dataset.